First we read the train and test folders and check the distributions



Notes:

->This code uses the train and test folder inside the dataset directory.

->These folders are taken from the Wifi_and_MQTT folder in the downloaded
CIC-IoMT-DIAD-2024 dataset.


In [ ]:
import os
import pandas as pd

# Change this to your dataset root
DATASET_ROOT = "..\Dataset"

def get_class_distribution(split_dir):
    """
    Count samples per CSV file inside split_dir (train or test).
    Returns a DataFrame with class name and number of samples.
    """
    rows = []
    for fname in os.listdir(split_dir):
        if fname.endswith(".csv"):
            class_name = os.path.splitext(fname)[0]  # filename without .csv
            fpath = os.path.join(split_dir, fname)
            try:
                # Load just enough rows to count quickly
                df = pd.read_csv(fpath)
                n_samples = len(df)
            except Exception as e:
                print(f"Error reading {fpath}: {e}")
                n_samples = 0
            rows.append({"class": class_name, "samples": n_samples})
    return pd.DataFrame(rows).sort_values("samples", ascending=False)

# Train split
train_dir = os.path.join(DATASET_ROOT, "train")
train_dist = get_class_distribution(train_dir)
print("Train distribution:")
print(train_dist)

# Test split
test_dir = os.path.join(DATASET_ROOT, "test")
test_dist = get_class_distribution(test_dir)
print("\nTest distribution:")
print(test_dist)

# Merge for global view
train_dist["split"] = "train"
test_dist["split"] = "test"
all_dist = pd.concat([train_dist, test_dist], ignore_index=True)

# Pivot for summary
summary = all_dist.pivot_table(values="samples", index="class", columns="split", fill_value=0)
summary["total"] = summary.sum(axis=1)

print("\nOverall distribution (train/test/total):")
print(summary)

# Save to CSV for later inspection
#summary.to_csv("class_distribution_summary.csv")

Then we categorize them into 6 different categories as mentioned by the dataset authors
1) Benign
2) DDoS
3) DoS
4) MQTT
5) Recon
6) Spoofing

In [ ]:
import pandas as pd


def map_group(class_name):
    c = class_name.lower()
    if "arp_spoofing" in c:
        return "Spoofing"
    elif "recon" in c:
        return "Recon"
    elif "mqtt" in c:
        return "MQTT"
    elif "ddos" in c:
        return "DDoS"
    elif "dos" in c:
        return "DoS"
    elif "benign" in c:
        return "Benign"
    else:
        return "Other"

# Apply mapping
train_dist["group"] = train_dist["class"].apply(map_group)
test_dist["group"] = test_dist["class"].apply(map_group)

# Summarize
train_grouped = train_dist.groupby("group")["samples"].sum().reset_index().sort_values("samples", ascending=False)
test_grouped = test_dist.groupby("group")["samples"].sum().reset_index().sort_values("samples", ascending=False)

# Merge
all_grouped = pd.merge(train_grouped, test_grouped, on="group", how="outer", suffixes=("_train", "_test")).fillna(0)
all_grouped["total"] = all_grouped["samples_train"] + all_grouped["samples_test"]

#print("Grouped distribution (train/test/total):")
#print(all_grouped)

# Save for later
#all_grouped.to_csv("grouped_class_distribution.csv", index=False)


Distribute them into Train, Test and Val Split as 80/10/10


Notes:

->This code creates a new folder named as final_split

-> final_split folder will contain 3 csv files named as train,test and val


In [5]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Paths
DATASET_ROOT = "..\Dataset"
train_dir = os.path.join(DATASET_ROOT, "train")
test_dir = os.path.join(DATASET_ROOT, "test")

# Define grouping rules
def map_group(class_name):
    c = class_name.lower()
    if "arp_spoofing" in c:
        return "Spoofing"
    elif "recon" in c:
        return "Recon"
    elif "mqtt" in c:
        return "MQTT"
    elif "ddos" in c:
        return "DDoS"
    elif "dos" in c:
        return "DoS"
    elif "benign" in c:
        return "Benign"
    else:
        return "Other"

def load_and_group(input_dir):
    all_dfs = []
    for fname in os.listdir(input_dir):
        if fname.endswith(".csv"):
            class_name = os.path.splitext(fname)[0]
            group = map_group(class_name)

            fpath = os.path.join(input_dir, fname)
            try:
                df = pd.read_csv(fpath)
            except Exception as e:
                print(f"Error reading {fpath}: {e}")
                continue

            # Add label column
            df["label"] = group
            all_dfs.append(df)

    if all_dfs:
        return pd.concat(all_dfs, ignore_index=True)
    else:
        return pd.DataFrame()

# Load train and test folders
train_df = load_and_group(train_dir)
test_df = load_and_group(test_dir)

# Combine everything
full_df = pd.concat([train_df, test_df], ignore_index=True)
print(f"Full dataset shape: {full_df.shape}")
print(full_df["label"].value_counts())

# Split into 80/10/10 (stratified by label)
train_df, temp_df = train_test_split(
    full_df, test_size=0.2, stratify=full_df["label"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42
)

print(f"Train size: {len(train_df)}")
print(f"Val size: {len(val_df)}")
print(f"Test size: {len(test_df)}")

# Save splits
output_dir = os.path.join(DATASET_ROOT, "final_split")
os.makedirs(output_dir, exist_ok=True)

train_df.to_csv(os.path.join(output_dir, "train.csv"), index=False)
val_df.to_csv(os.path.join(output_dir, "val.csv"), index=False)
test_df.to_csv(os.path.join(output_dir, "test.csv"), index=False)

print(f"Saved train/val/test CSVs to {output_dir}")



Full dataset shape: (9162994, 40)
label
DDoS        5846623
DoS         2222205
MQTT         883951
Recon        131402
Benign        56898
Spoofing      21915
Name: count, dtype: int64
Train size: 7330395
Val size: 916299
Test size: 916300
Saved train/val/test CSVs to ..\Dataset\final_split


Analyze and Validate the data distribution across the splits


Notes:

->This code creates csv file inside the Preprocessing folder that will contain the class distributions.

In [6]:
import os
import pandas as pd

# Path where you saved the final splits
DATASET_ROOT = r"..\Dataset\final_split"

# Function to check distribution
def check_distribution(file_path, split_name):
    df = pd.read_csv(file_path)
    counts = df["label"].value_counts().reset_index()
    counts.columns = ["class", "samples"]
    counts["split"] = split_name
    return counts

# Load distributions
train_dist = check_distribution(os.path.join(DATASET_ROOT, "train.csv"), "train")
val_dist   = check_distribution(os.path.join(DATASET_ROOT, "val.csv"), "val")
test_dist  = check_distribution(os.path.join(DATASET_ROOT, "test.csv"), "test")

# Combine
all_dist = pd.concat([train_dist, val_dist, test_dist], ignore_index=True)

# Pivot for summary
summary = all_dist.pivot_table(values="samples", index="class", columns="split", fill_value=0)
summary["total"] = summary.sum(axis=1)

print("Class distribution across splits:")
print(summary)

# Save for later inspection
summary.to_csv(os.path.join(".\\", "class_distribution_summary.csv"))


Class distribution across splits:
split         test      train       val      total
class                                             
Benign      5690.0    45518.0    5690.0    56898.0
DDoS      584663.0  4677298.0  584662.0  5846623.0
DoS       222221.0  1777764.0  222220.0  2222205.0
MQTT       88395.0   707161.0   88395.0   883951.0
Recon      13140.0   105122.0   13140.0   131402.0
Spoofing    2191.0    17532.0    2192.0    21915.0


Preprocessing

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import os



#  Define Paths & Load Data
DATASET_ROOT = r"..\Dataset\final_split" # Use your path
train_file = os.path.join(DATASET_ROOT, "train.csv")
val_file   = os.path.join(DATASET_ROOT, "val.csv")
test_file  = os.path.join(DATASET_ROOT, "test.csv")

# Load datasets
train_df = pd.read_csv(train_file)
val_df   = pd.read_csv(val_file)
test_df  = pd.read_csv(test_file)

#  Separate Labels and Features
y_train = train_df["label"].copy()
y_val   = val_df["label"].copy()
y_test  = test_df["label"].copy()


# Drop the label
X_train = train_df.drop(columns=["label"])
X_val   = val_df.drop(columns=["label"])
X_test  = test_df.drop(columns=["label"])

# Fix infinities
for df in [X_train, X_val, X_test]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)


# Features that are continuous and need scaling
continuous_features = [
    'Header_Length', 'Time_To_Live', 'Rate', 'ack_count', 'syn_count',
    'fin_count', 'rst_count', 'Tot sum', 'Min', 'Max', 'AVG', 'Std',
    'Tot size', 'IAT', 'Number', 'Variance'
]

# Features that are binary/flags/proportions
binary_features = [
    'fin_flag_number', 'syn_flag_number', 'rst_flag_number',
    'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number',
    'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC', 'TCP', 'UDP',
    'DHCP', 'ARP', 'ICMP', 'IGMP', 'IPv', 'LLC'
]

# Categorical Feature
categorical_features = ['Protocol Type']



# Pipeline for CONTINUOUS features
continuous_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])


# Pipeline for BINARY/FLAG features
binary_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent'))
])


# Pipeline for the ONE CATEGORICAL feature
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])


#  Combine all pipelines with ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('continuous', continuous_pipeline, continuous_features),
        ('binary', binary_pipeline, binary_features),
        ('categorical', categorical_pipeline, categorical_features)
    ],
    remainder='passthrough' # Keep any columns we might have missed (just in case)
)



# The output is a NumPy array, ready for the model
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed   = preprocessor.transform(X_val)
X_test_processed  = preprocessor.transform(X_test)



#  Encode labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded   = label_encoder.transform(y_val)
y_test_encoded  = label_encoder.transform(y_test)



# Results
print(f"Original feature count: {len(X_train.columns)}")
print(f"Processed feature count: {X_train_processed.shape[1]} (due to OHE)")
print("\nShapes after preprocessing:")
print("Train:", X_train_processed.shape, "Labels:", y_train_encoded.shape)
print("Val:  ", X_val_processed.shape, "Labels:", y_val_encoded.shape)
print("Test: ", X_test_processed.shape, "Labels:", y_test_encoded.shape)
print("\nLabel mapping:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))


Original feature count: 39
Processed feature count: 43 (due to OHE)

Shapes after preprocessing:
Train: (7330395, 43) Labels: (7330395,)
Val:   (916299, 43) Labels: (916299,)
Test:  (916300, 43) Labels: (916300,)

Label mapping: {'Benign': np.int64(0), 'DDoS': np.int64(1), 'DoS': np.int64(2), 'MQTT': np.int64(3), 'Recon': np.int64(4), 'Spoofing': np.int64(5)}
